# Day 082 — Exercise 2: Long-Term Memory (SQLite)

**What you'll build:** `LongTermMemory` — a durable key/value store, backed by SQLite, that survives a restart.

**Why it matters:** working memory vanishes when the session ends. For an agent to remember *you* — your name, your preferences, your goals — it needs storage that persists. SQLite (stdlib, zero setup) is the workhorse: point it at a file and the facts are still there next run.

In [ ]:
# Long-term memory persists facts in SQLite (sqlite3 is stdlib).


## Task

`LongTermMemory(db_path=":memory:")` — connect and `CREATE TABLE IF NOT EXISTS memories(key TEXT PRIMARY KEY, value TEXT)`.

- `remember(key, value)` — `INSERT OR REPLACE` (upsert); return `self`.
- `recall(key)` — the value, or `None`.
- `search(term)` — `[{key, value}]` where `term` is in key or value (case-insensitive).
- `all()` — every fact as `[{key, value}]`, ordered by key.
- `forget(key)` — delete; return `self`. `__len__` — row count. `close()` — close the connection.

Pass a file path (not `:memory:`) and a **new** `LongTermMemory` on that path sees what an earlier one stored — that's persistence.

## Your Implementation

In [ ]:
class LongTermMemory:
    """Durable key/value memory backed by SQLite (persists across sessions)."""

    def __init__(self, db_path=":memory:"):
        raise NotImplementedError

    def remember(self, key, value):
        raise NotImplementedError

    def recall(self, key):
        raise NotImplementedError

    def search(self, term):
        raise NotImplementedError

    def all(self):
        raise NotImplementedError

    def forget(self, key):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def close(self):
        raise NotImplementedError


In [ ]:

# ── long-term memory (persists across sessions, SQLite) ──────────────────────
import sqlite3


class LongTermMemory:
    """Durable key/value memory backed by SQLite - survives restarts.

    Default db_path is ":memory:" (a private in-process database). Pass a file
    path to persist across sessions: a new LongTermMemory on the same path sees
    everything a previous one remembered.
    """

    def __init__(self, db_path=":memory:"):
        self.db_path = db_path
        self._conn = sqlite3.connect(db_path)
        self._conn.execute(
            "CREATE TABLE IF NOT EXISTS memories "
            "(key TEXT PRIMARY KEY, value TEXT NOT NULL)"
        )
        self._conn.commit()

    def remember(self, key, value):
        """Store or overwrite a fact by key. Returns self."""
        self._conn.execute(
            "INSERT OR REPLACE INTO memories(key, value) VALUES(?, ?)",
            (str(key), str(value)))
        self._conn.commit()
        return self

    def recall(self, key):
        """Return the stored value for a key, or None if unknown."""
        row = self._conn.execute(
            "SELECT value FROM memories WHERE key = ?", (str(key),)).fetchone()
        return row[0] if row else None

    def search(self, term):
        """Return [{key, value}] where term (case-insensitive) is in key or value."""
        term = str(term).lower()
        rows = self._conn.execute("SELECT key, value FROM memories").fetchall()
        return [{"key": k, "value": v} for k, v in rows
                if term in k.lower() or term in v.lower()]

    def all(self):
        """Return every fact as [{key, value}], ordered by key."""
        rows = self._conn.execute(
            "SELECT key, value FROM memories ORDER BY key").fetchall()
        return [{"key": k, "value": v} for k, v in rows]

    def forget(self, key):
        """Delete a fact by key. Returns self."""
        self._conn.execute("DELETE FROM memories WHERE key = ?", (str(key),))
        self._conn.commit()
        return self

    def __len__(self):
        return self._conn.execute("SELECT COUNT(*) FROM memories").fetchone()[0]

    def close(self):
        """Close the database connection."""
        self._conn.close()


## Automated checks

In [ ]:

import os, tempfile
score, total = 0, 6
try:
    m = LongTermMemory()      # ":memory:" - private in-process db
    m.remember('name', 'Kutlwano').remember('lang', 'Python')
    assert len(m) == 2 and m.recall('name') == 'Kutlwano'
    score += 1; print("✅ remember/recall store and fetch facts")

    m.remember('name', 'Kutlwano M.')     # same key overwrites
    assert m.recall('name') == 'Kutlwano M.' and len(m) == 2
    score += 1; print("✅ remember overwrites an existing key (upsert)")

    assert m.recall('missing') is None
    score += 1; print("✅ recall returns None for an unknown key")

    hits = m.search('python')
    assert len(hits) == 1 and hits[0]['key'] == 'lang'
    score += 1; print("✅ search matches key or value, case-insensitively")

    m.forget('lang')
    assert m.recall('lang') is None and len(m) == 1
    score += 1; print("✅ forget deletes a fact")

    path = tempfile.NamedTemporaryFile(suffix='.db', delete=False).name
    try:
        a = LongTermMemory(path); a.remember('goal', 'ship agents'); a.close()
        b = LongTermMemory(path)
        assert b.recall('goal') == 'ship agents'
        b.close()
        score += 1; print("✅ long-term memory persists across sessions (SQLite file)")
    finally:
        os.unlink(path)

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── long-term memory (persists across sessions, SQLite) ──────────────────────
import sqlite3


class LongTermMemory:
    """Durable key/value memory backed by SQLite - survives restarts.

    Default db_path is ":memory:" (a private in-process database). Pass a file
    path to persist across sessions: a new LongTermMemory on the same path sees
    everything a previous one remembered.
    """

    def __init__(self, db_path=":memory:"):
        self.db_path = db_path
        self._conn = sqlite3.connect(db_path)
        self._conn.execute(
            "CREATE TABLE IF NOT EXISTS memories "
            "(key TEXT PRIMARY KEY, value TEXT NOT NULL)"
        )
        self._conn.commit()

    def remember(self, key, value):
        """Store or overwrite a fact by key. Returns self."""
        self._conn.execute(
            "INSERT OR REPLACE INTO memories(key, value) VALUES(?, ?)",
            (str(key), str(value)))
        self._conn.commit()
        return self

    def recall(self, key):
        """Return the stored value for a key, or None if unknown."""
        row = self._conn.execute(
            "SELECT value FROM memories WHERE key = ?", (str(key),)).fetchone()
        return row[0] if row else None

    def search(self, term):
        """Return [{key, value}] where term (case-insensitive) is in key or value."""
        term = str(term).lower()
        rows = self._conn.execute("SELECT key, value FROM memories").fetchall()
        return [{"key": k, "value": v} for k, v in rows
                if term in k.lower() or term in v.lower()]

    def all(self):
        """Return every fact as [{key, value}], ordered by key."""
        rows = self._conn.execute(
            "SELECT key, value FROM memories ORDER BY key").fetchall()
        return [{"key": k, "value": v} for k, v in rows]

    def forget(self, key):
        """Delete a fact by key. Returns self."""
        self._conn.execute("DELETE FROM memories WHERE key = ?", (str(key),))
        self._conn.commit()
        return self

    def __len__(self):
        return self._conn.execute("SELECT COUNT(*) FROM memories").fetchone()[0]

    def close(self):
        """Close the database connection."""
        self._conn.close()
```

**Why `INSERT OR REPLACE`?** Memory is keyed by a stable name — `name`, `goal`. When a fact changes you want to overwrite, not accumulate duplicates. `INSERT OR REPLACE` on a `PRIMARY KEY` column is SQLite's one-line upsert: new key inserts, existing key overwrites.

</details>